# Phase 3 RQ1 v2 ? B0 training: seed 44, order-1

**This notebook runs exactly one seed/order pair: seed `42`, `order-1`.** Use a Kaggle GPU/CUDA runtime, turn Internet on, attach this run folder?s input dataset, accept the gated `meta-llama/Llama-3.2-1B` terms, and configure the Kaggle secret `HF_TOKEN`.

The workflow validates the frozen input protocol and XNLI data hash before training. Download and retain the output `phase3_rq1_v2_b0_seed_44_order-1.zip`; this is a training-result ZIP and is different from the input ZIP attached here.

The input bundle uses the Phase 4 roster `en`, `de`, `es`, `fr`, `sw`, `ar`, `ur`, `bg`, `ru`, `hi`. Arabic published-interference values are imputed using the recorded URIEL+ procedure; Grambank is excluded for coverage.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import uuid
from zipfile import ZIP_DEFLATED, ZipFile

REPO_URL = "https://github.com/Mani-S-2709/asymmetric-interference-continual-learning..git"
REPO_ROOT = Path("/kaggle/working/asymmetric-interference-continual-learning")
if not REPO_ROOT.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_ROOT)], check=True)
elif (REPO_ROOT / ".git").is_dir():
    subprocess.run(["git", "-C", str(REPO_ROOT), "remote", "set-url", "origin", REPO_URL], check=True)
    subprocess.run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only"], check=True)
else:
    raise FileExistsError(f"Repository path exists but is not a Git clone: {REPO_ROOT}")

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "-r",
    str(REPO_ROOT / "phase2" / "requirements-phase2.txt"),
], check=True)

from kaggle_secrets import UserSecretsClient
hf_token = UserSecretsClient().get_secret("HF_TOKEN")
if not hf_token:
    raise RuntimeError("HF_TOKEN Kaggle secret is empty.")
os.environ["HF_TOKEN"] = hf_token
del hf_token

sys.path.insert(0, str(REPO_ROOT))
input_bundles = list(Path("/kaggle/input").rglob("phase3_rq1_v2_inputs.zip"))
if not input_bundles:
    input_summaries = list(Path("/kaggle/input").rglob("rq1_v2_input_summary.json"))
    if len(input_summaries) == 1:
        input_root = input_summaries[0].parent
        input_bundle = Path("/kaggle/working/phase3_rq1_v2_inputs.zip")
        with ZipFile(input_bundle, "w", compression=ZIP_DEFLATED) as archive:
            for path in sorted(input_root.rglob("*")):
                if path.is_file():
                    archive.write(path, path.relative_to(input_root).as_posix())
        input_bundles = [input_bundle]
        print(f"Repacked the extracted input dataset as: {input_bundle}")
if len(input_bundles) != 1:
    raise FileNotFoundError(
        "Attach exactly one Kaggle dataset containing either "
"phase3_rq1_v2_inputs.zip or its extracted contents. Found ZIPs: "
+ str(input_bundles)
    )

SEED_TO_RUN = 44
if SEED_TO_RUN not in (42, 43, 44):
    raise ValueError("SEED_TO_RUN must be 42, 43, or 44.")
ORDER_TO_RUN = "order-1"
if ORDER_TO_RUN not in {f"order-{index}" for index in range(1, 6)}:
    raise ValueError("ORDER_TO_RUN must be order-1 through order-5.")

OUTPUT_DIR = Path("/kaggle/working/phase3_rq1_v2_seed_outputs")
WORK_DIR = Path("/kaggle/temp/phase3_rq1_v2") / f"seed_{SEED_TO_RUN}_{uuid.uuid4().hex}"
subprocess.run([
    sys.executable, "-m", "phase3.rq1_v2_workflow", "train-order",
    "--inputs-zip", str(input_bundles[0]),
    "--seed", str(SEED_TO_RUN),
    "--order-id", ORDER_TO_RUN,
    "--work-dir", str(WORK_DIR),
    "--output-dir", str(OUTPUT_DIR),
], cwd=REPO_ROOT, check=True)

RESULT_ZIP = OUTPUT_DIR / f"phase3_rq1_v2_b0_seed_{SEED_TO_RUN}_{ORDER_TO_RUN}.zip"
if not RESULT_ZIP.is_file():
    raise FileNotFoundError(f"Validated result bundle was not created: {RESULT_ZIP}")
print(f"Download this seed's result bundle from Kaggle output files: {RESULT_ZIP}")